In [ ]:
# 📌 1. Imports
import os
import cv2
import numpy as np
from skimage.metrics import structural_similarity as ssim
from tqdm import tqdm


In [3]:
# 📌 2. SSIM function
def calculate_ssim(img1, img2):
    # Convert to grayscale (SSIM is usually applied on luminance channel)
    img1_gray = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
    img2_gray = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)
    
    score, _ = ssim(img1_gray, img2_gray, full=True)
    return score



In [4]:
# 📌 3. Function to compute SSIM per class
def compute_ssim_for_class(class_folder):
    real_images = [f for f in os.listdir(class_folder) if f.startswith("ISIC")]
    fake_images = [f for f in os.listdir(class_folder) if not f.startswith("ISIC")]
    
    ssim_scores = []
    
    for real, fake in tqdm(zip(real_images, fake_images), total=min(len(real_images), len(fake_images))):
        real_path = os.path.join(class_folder, real)
        fake_path = os.path.join(class_folder, fake)
        
        # load and resize to same size
        img1 = cv2.imread(real_path)
        img2 = cv2.imread(fake_path)
        
        if img1 is None or img2 is None:
            continue
        
        img2 = cv2.resize(img2, (img1.shape[1], img1.shape[0]))
        
        ssim_val = calculate_ssim(img1, img2)
        ssim_scores.append(ssim_val)
    
    return np.mean(ssim_scores) if ssim_scores else None


In [5]:
# 📌 4. Compute SSIM for all classes
base_path = "C:\\Users\\devgo\\OneDrive\\Desktop\\fid\\data\\processed_images"
classes = os.listdir(base_path)

results = {}
all_scores = []

for cls in classes:
    cls_path = os.path.join(base_path, cls)
    if not os.path.isdir(cls_path):
        continue
    
    ssim_val = compute_ssim_for_class(cls_path)
    if ssim_val is not None:
        results[cls] = ssim_val
        all_scores.append(ssim_val)

# Overall average SSIM
results["Overall"] = np.mean(all_scores)

results


100%|██████████| 1113/1113 [00:00<00:00, 2075.64it/s]
0it [00:00, ?it/s]
0it [00:00, ?it/s]
100%|██████████| 142/142 [00:00<00:00, 1975.17it/s]


{'AKIEC': np.float64(0.30207821111345057),
 'BCC': np.float64(0.33332108829695917),
 'BKL': np.float64(0.3411753167997013),
 'DF': np.float64(0.4384945709376715),
 'MEL': np.float64(0.3189929478926417),
 'VASC': np.float64(0.4503730476270433),
 'Overall': np.float64(0.36407253044457794)}